## Берем датасет Reuters-21578 и достаем из него заголовки                                   

In [ ]:
%pip install -r requirements.txt

In [3]:
import numpy as np
import pandas as pd

In [4]:
import html
import nltk
from nltk.corpus import reuters

nltk.download('reuters')

def load_reuters(split):
    ids = [f for f in reuters.fileids() if f.startswith(split + '/')]
    # ids = ['earn', 'acq', 'crude', 'trade', 'money-fx', 'interest', 'ship']
    texts = [html.unescape(reuters.raw(f)) for f in ids]
    labels = [reuters.categories(f) for f in ids]
    return texts, labels

def clean_text(text):
    text = html.unescape(text)
    title, _, body = text.partition('\n')
    text = f'{title.strip()}. {body}'
    text = TICKER.sub(' ', text)
    text = re.sub(r'[<>]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

[nltk_data] Downloading package reuters to
[nltk_data]     C:\Users\382he\AppData\Roaming\nltk_data...


In [11]:
raw_train_texts, raw_train_labels = load_reuters('training')
raw_test_texts, raw_test_labels = load_reuters('test')

In [12]:
import re
TICKER = re.compile(r'<[A-Z]{1,5}(?:\.[A-Z]{1,3})*>')

text_train = [clean_text(t) for t in raw_train_texts]
text_test  = [clean_text(t) for t in raw_test_texts]

def single_label(texts, labels):
    pairs = [(t, l[0]) for t, l in zip(texts, labels) if len(l) == 1]
    return [t for t, _ in pairs], [l for _, l in pairs]

text_train1, tags_train1 = single_label(text_train, raw_train_labels)
text_test1,  tags_test1  = single_label(text_test, raw_test_labels)

In [13]:
print(len(text_train), len(text_test))

7769 3019


## Получение эмбеддингов (Bag Of Words)

In [15]:
from sklearn.feature_extraction.text import CountVectorizer

# Полный BoW без фильтрации словаря. Матрица хранится в разреженном формате CSR:
# в памяти лежат только ненулевые счётчики (~0.3% ячеек), поэтому весь корпус занимает единицы МБ.
# Все модели sklearn (TruncatedSVD, LogisticRegression, KNN, KMeans...) принимают такую матрицу напрямую.
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(text_train1)   # fit только на train
X_test_bow  = vectorizer.transform(text_test1)        # слова, которых нет в словаре train, отбрасываются
vocab = vectorizer.get_feature_names_out()

In [17]:
# Табличный вид BoW: разреженный DataFrame, колонки — слова, строки — документы.
# Обычный pd.DataFrame(X.toarray()) развернул бы все нули и занял ~1.2 ГБ.
bow_df = pd.DataFrame.sparse.from_spmatrix(
    X_train_bow,
    columns=vocab,
    index=[f'{i+1}' for i in range(X_train_bow.shape[0])],
)
print(bow_df.shape, f'{bow_df.memory_usage().sum() / 2**20:.1f} МБ')

# слова первого документа по убыванию частоты
doc = bow_df.iloc[0]
doc[doc > 0].sort_values(ascending=False).head(16)

(6577, 21551) 5.1 МБ


and      23
at       20
the      18
to       14
dlrs     14
for      10
new       9
were      9
york      8
of        8
sales     7
cocoa     7
times     7
in        6
dec       6
are       5
Name: 1, dtype: Sparse[int64, 0]

In [18]:
# Если всё же нужна плотная матрица (numpy-операции, методы без поддержки sparse):
# счётчики маленькие, поэтому хватает uint8 — 1 байт на ячейку вместо 8 (~150 МБ вместо ~1.2 ГБ).
assert X_train_bow.max() <= np.iinfo(np.uint8).max
X_train_dense = X_train_bow.astype(np.uint8).toarray()
print(X_train_dense.shape, f'{X_train_dense.nbytes / 2**20:.0f} МБ')

(6577, 21551) 135 МБ


In [20]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import Normalizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import validation_curve
import numpy as np

pipe = make_pipeline(CountVectorizer(min_df=2, stop_words="english"),
                     Normalizer(), LinearSVC(class_weight="balanced"))
Cs = np.logspace(-3, 2, 11)
tr, va = validation_curve(pipe, text_train1, tags_train1, param_name="linearsvc__C",
                          param_range=Cs, cv=5, scoring="f1_macro")
# рисуй tr.mean(1) и va.mean(1) от Cs с plt.semilogx

c:\Users\382he\nlp\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\382he\nlp\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\382he\nlp\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\382he\nlp\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\382he\nlp\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\382he\nlp\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge